In [54]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [55]:
import warnings
pd.options.mode.chained_assignment = None
warnings.filterwarnings('ignore')

In [56]:
df = pd.read_csv('source/FReDA4.csv')

In [57]:
# df2 = df.dropna(subset=traits).copy()
# print(df2['Group3'].value_counts())

In [58]:
df = df.rename(columns={
    'Work Status': 'Work_Status',

    'Self-esteem': 'Self_esteem',
    'Life Satisfaction': 'Life_Satisfaction',

    'Relationship Sex': 'Relationship_Sex',
    'Relationship Length': 'Relationship_Length',
    'Age difference': 'Age_Difference',
    'Communication Quality': 'Communication_Quality',
    'Relationship Satisfaction': 'Relationship_Satisfaction',
    'Conflict Management': 'Conflict_Management'
})

In [59]:
traits = [
    'Sex',
    'Age',
    'Work_Status',
    'Education',
    'Urbanization',

    'Extraversion',
    'Agreeableness',
    'Conscientiousness',
    'Openness',
    'Neuroticism',
    'Conservatism',
    'Religiosity',

    'Depressiveness',
    'Loneliness',
    'Self_esteem',
    'Life_Satisfaction',
    'Health',

    'Relationship_Sex',
    'Relationship_Length',
    'Age_Difference',
    'Married',
    'Cohabitation',
    'Kids',
    'Communication_Quality',
    'Relationship_Satisfaction',
    'Conflict_Management',
]

In [60]:
print(len(traits))

26


In [61]:
# 1. (1 = Satisfied, 0 = All others)
df['Is_Satisfied'] = np.where(df['Group3'] == 'Couple Satisfaction', 1, 0)

In [62]:
potential_predictors = traits
significant_predictors = []

In [63]:
for var in potential_predictors:
    # Check bivariate association
    formula = f"Is_Satisfied ~ {var}"

    model = smf.gee(formula,
                    groups=df['CoupleId'],
                    data=df,
                    family=sm.families.Binomial()
                    ).fit()

    # If p-value of the predictor is significant (< 0.05), keep it
    p_val = model.pvalues.iloc[1]
    if p_val < 0.05:
        significant_predictors.append(var)

In [64]:
print('Significant predictors found:', len(significant_predictors))
print(significant_predictors)

Significant predictors found: 19
['Age', 'Work_Status', 'Extraversion', 'Agreeableness', 'Conscientiousness', 'Openness', 'Neuroticism', 'Depressiveness', 'Loneliness', 'Self_esteem', 'Life_Satisfaction', 'Health', 'Relationship_Length', 'Married', 'Cohabitation', 'Kids', 'Communication_Quality', 'Relationship_Satisfaction', 'Conflict_Management']


In [65]:
final_formula = "Is_Satisfied ~ " + " + ".join(significant_predictors)

# Using GEE with a Binomial family acts as a logistic regression that adjusts StdErrs for couples
final_model = smf.gee(final_formula,
                      groups=df['CoupleId'],
                      data=df,
                      family=sm.families.Binomial()
                      ).fit()

In [66]:
# 4. Adjusted Odds Ratios (AOR) and 95% Confidence Intervals
results_df = pd.DataFrame({
    'AOR': np.exp(final_model.params),
    'Lower CI': np.exp(final_model.conf_int()[0]),
    'Upper CI': np.exp(final_model.conf_int()[1]),
    'p-value': final_model.pvalues
})
print(results_df)


                                AOR  Lower CI  Upper CI       p-value
Intercept                  0.001724  0.000673  0.004418  4.408336e-40
Age                        1.015701  1.007115  1.024359  3.220493e-04
Work_Status                1.144168  1.071742  1.221488  5.422532e-05
Extraversion               0.980780  0.958630  1.003441  9.587236e-02
Agreeableness              0.976698  0.954402  0.999514  4.537169e-02
Conscientiousness          0.989498  0.968545  1.010903  3.336077e-01
Openness                   1.003139  0.981178  1.025593  7.813683e-01
Neuroticism                0.994947  0.969907  1.020633  6.968570e-01
Depressiveness             0.996307  0.955427  1.038936  8.625898e-01
Loneliness                 0.878830  0.830156  0.930358  8.869304e-06
Self_esteem                1.033977  1.007288  1.061374  1.227184e-02
Life_Satisfaction          1.004060  0.965836  1.043795  8.378961e-01
Health                     1.076807  1.007639  1.150722  2.891599e-02
Relationship_Length 